# 11. Saving the model and predicting new people

Every time `main.py` finishes, it refits the best model on all development rows... and throws it away. Today we keep it:
1. `main.py` **saves** the fitted pipeline to a file;
2. a new script, `predict.py`, **loads** it and predicts a new CSV from the terminal, with no training.

## 1 · Setup

Same `course_helpers.py` as the other notebook. We fit the pipeline on all development rows, as `main.py` does at the end of a run.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from course_helpers import *        # last week's pipeline: build_pipeline, clean_dataset, CONFIG, ...

X_dev, y_dev, extras_dev, cv = load_course_data()
pipeline = build_pipeline(CONFIG["preprocessing"], CONFIG["model"], CONFIG["feature_selection"]).fit(X_dev, y_dev)
print("fitted pipeline:", " -> ".join(pipeline.named_steps))

fitted pipeline: prep -> select -> model


## 2 · Save, then load

In [2]:
os.makedirs("models", exist_ok=True)
joblib.dump(pipeline, "models/model.joblib")          # save
loaded = joblib.load("models/model.joblib")            # load (e.g. in another script, another day)

print(f"same predictions after loading: {np.allclose(loaded.predict_proba(X_dev), pipeline.predict_proba(X_dev))}")
print("medians it remembers:", loaded.named_steps["prep"].named_transformers_["numeric"].named_steps["impute"].statistics_)
print("features it kept:", list(loaded.named_steps["select"].get_feature_names_out(loaded.named_steps["prep"].get_feature_names_out())))

same predictions after loading: True
medians it remembers: [31.  0.  0.  0.  2.]
features it kept: ['numeric__age', 'numeric__priors_count']


The file holds the **whole fitted Pipeline**: every value learned in training. A new person goes through exactly the same steps, with those values. Nothing is re-learned on the new data.

<div style="background-color:#fdecea; border-left:6px solid #c62828; padding:10px 14px; border-radius:4px">⚠️ Load a model with the <b>same library versions</b> it was saved with (scikit-learn, XGBoost), and only load files you trust: a <code>.joblib</code> file can run code when loaded.</div>

## 3 · Predicting new people

`data/compas_new_people.csv`: 20 people the model has never seen, **without** the target. Like real new data, it isn't clean:

In [3]:
new_people = pd.read_csv("data/compas_new_people.csv")
new_people.loc[[3, 7, 11, 14, 15], ["id", "age", "priors_count", "c_charge_degree", "sex"]]

,id,age,priors_count,c_charge_degree,sex
3,900004,22.0,?,M,Male
7,900008,NaN,3.0,M,Female
11,900012,25.0,14.0,X,Male
14,900015,36.0,4.0,M,Male
15,900016,36.0,4.0,M,Male


<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:10px 14px; border-radius:4px">🤔 <b>Guess first:</b> row 3 has <code>priors_count = "?"</code>, row 7 no age, row 11 a charge degree <code>"X"</code> never seen in training, and rows 14–15 are the same person twice. How many predictions should come out?</div>

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:10px 14px; border-radius:4px"><b>🆕 New: <code>predict.py</code></b> (next to <code>main.py</code>)<br>Cleans the new file with the same rules as training (<b>without</b> <code>drop_duplicate_rows()</code>: every row gets an answer), then uses the saved pipeline.</div>

In [4]:
def predict(df: pd.DataFrame, pipeline, config: dict) -> pd.DataFrame:
    """Returns the id, the probability of the positive class and the prediction (at 0.5) for every row."""
    clean = clean_dataset(df, config["diagnostics"])
    X, _, _ = split_features_target(clean, config["data"], config["preprocessing"]["mnar_indicator_sources"])
    probability = pipeline.predict_proba(X)[:, 1]
    return pd.DataFrame({"id": df[config["diagnostics"]["id_column"]],
                         "probability": probability,
                         "prediction": (probability >= 0.5).astype(int)})

In [5]:
predictions = predict(new_people, joblib.load("models/model.joblib"), CONFIG)
print(f"{len(new_people)} rows in, {len(predictions)} rows out")
predictions.loc[[3, 7, 11, 14, 15]]

20 rows in, 20 rows out


,id,probability,prediction
3,900004,0.664798,1
7,900008,0.544961,1
11,900012,0.889861,1
14,900015,0.470575,0
15,900016,0.470575,0


- Rows 3 and 7: the cleaning turns `"?"` into a gap, and the saved imputer fills both gaps.
- Row 11: our encoders accept categories never seen in training, so `"X"` doesn't crash anything.
- Rows 14 and 15: same person, same prediction, **both kept**.

## 4 · In the pipeline

**`main.py`**: at the end, after the final refit (`import os`, `import joblib` at the top):
```python
    # The final fitted pipeline: what predict.py loads to predict new data.
    model_path = config.get("output", {}).get("model_path", "models/model.joblib")
    os.makedirs(os.path.dirname(model_path) or ".", exist_ok=True)
    pipeline.set_params(memory=None)                # the cache folder was deleted; the saved model doesn't need it
    joblib.dump(pipeline, model_path)
```
**`config.yaml`**, in `output`:
```yaml
  model_path: "models/model.joblib"                 # the final fitted pipeline, used by predict.py (overwritten each run)
```
**`predict.py`**: the `predict()` function above, plus a few lines to run it from the terminal:
```python
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Predict a CSV with the model saved by main.py.")
    parser.add_argument("input", help="CSV with the same columns as the training data (no target needed)")
    parser.add_argument("output", nargs="?", default="predictions.csv", help="where to write the predictions")
    args = parser.parse_args()

    with open("config.yaml") as f:
        config = yaml.safe_load(f)
    pipeline = joblib.load(config["output"]["model_path"])
    predictions = predict(pd.read_csv(args.input), pipeline, config)
    predictions.to_csv(args.output, index=False)
    print(f"{len(predictions)} rows predicted -> {args.output}")
```
From the project folder (macOS, Linux and Windows):
```bash
python main.py                                                   # train, evaluate, save models/model.joblib
python predict.py data/compas_new_people.csv predictions.csv     # predict
```

## Challenges (in your own pipeline)

**Part A is required**: it's what the weekly check looks for. Copy the code from the two notebooks: every pipeline function in them is exactly the course pipeline's version.

<div style="background-color:#ede7f6; border-left:6px solid #5e35b1; padding:10px 14px; border-radius:4px"><b>A1 · The imputer as a setting</b> (<code>10_missing_values</code>, section 4)<br><ul><li><code>src/preprocessing.py</code>: the new imports, <code>_NUMERIC_IMPUTERS</code> and the new <code>build_preprocessor()</code>.</li><li><code>config.yaml</code>: the new <code>imputation</code> options.</li><li>Run <code>median</code>, <code>knn</code>, <code>iterative</code> and <code>random_forest</code> (tuning off is fine) and compare them with <code>corrected_paired_test()</code>.</li></ul></div>

<div style="background-color:#ede7f6; border-left:6px solid #5e35b1; padding:10px 14px; border-radius:4px"><b>A2 · Save and predict</b> (this notebook, section 4)<br><ul><li><code>main.py</code> saves the pipeline; <code>config.yaml</code>: <code>output.model_path</code>; <code>.gitignore</code>: <code>models/</code>.</li><li>New <code>predict.py</code>; copy <code>data/compas_new_people.csv</code> into your <code>data/</code>.</li><li><b>Check:</b> <code>python predict.py data/compas_new_people.csv predictions.csv</code> writes 20 rows.</li></ul></div>

### README task (due Saturday 23:59)
- Add the W10 row to "Pipeline progress".
- **Missing values:** your imputer runs and the corrected test; which imputer do you keep, and why? (2–3 sentences)
- **Predicting new data:** the command, and what the output contains.

### Part B · Extra challenges

<div style="background-color:#fce4ec; border-left:6px solid #ad1457; padding:10px 14px; border-radius:4px"><b>B1 · Multiple imputation.</b> Train <i>m</i> pipelines with <code>IterativeImputer(sample_posterior=True, random_state=i)</code> and average their probabilities (<code>10_missing_values</code>, section 6). Is it worth <i>m</i> times the cost?</div>

<div style="background-color:#fce4ec; border-left:6px solid #ad1457; padding:10px 14px; border-radius:4px"><b>B2 · Let the imputer see the categories.</b> <code>age_cat</code> halved the error on <code>age</code> (<code>10_missing_values</code>, section 5). Change the preprocessing so the model-based imputers also see the categorical columns. Do the guesses improve? The model's score?</div>

<div style="background-color:#fce4ec; border-left:6px solid #ad1457; padding:10px 14px; border-radius:4px"><b>B3 · Random-forest proximities, from scratch.</b> <code>forest.apply(X)</code> gives each row's leaf in every tree; two rows' proximity = the share of trees where they share a leaf. Fill the gaps of <code>priors_count</code> with the proximity-weighted average and compare with <code>random_forest</code> on hidden values. Could this run in <code>predict.py</code>?</div>

<div style="background-color:#fce4ec; border-left:6px solid #ad1457; padding:10px 14px; border-radius:4px"><b>B4 · A safer predict.py.</b> Stop with a clear message when the input lacks a column the model needs; save the library versions next to the model; a small test that checks rows out = rows in.</div>

## Week 10 summary: everything added to the pipeline this week

| File | Change |
|---|---|
| `src/preprocessing.py` | `_NUMERIC_IMPUTERS` (median, mean, knn, iterative, random_forest); `build_preprocessor()` scales, then imputes |
| `main.py` | saves the final pipeline to `models/model.joblib`; imputer on the scoreboard |
| `predict.py` (new) | predicts a CSV from the terminal, one row per input row |
| `config.yaml` | `imputation` options; `output.model_path` |
| `src/results.py` | imputer in the report header |
| `.gitignore`, `requirements.txt`, `data/` | `models/`, `predictions.csv`; `joblib`; `compas_new_people.csv` |

**By Saturday 23:59** your fork should have: Part A done and the README task written.